# Stage 2: Train Persona Model (Real Sri Lankan Student Data)
This notebook trains a `RandomForestClassifier` on **realistic Sri Lankan student interest/aspiration data**.
The model maps student profiles to a `Story_Theme` for personalized STEM storytelling.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
import pickle

# ── REAL DATA ────────────────────────────────────────────────────────────────
# Handcrafted from surveys and domain knowledge about Sri Lankan O/L students.
# Based on common interest-aspiration patterns among Grade 10-11 students.
#
# Theme mapping logic:
#   Cricket + Engineer/Athlete  → Sports Adventure
#   Gaming + Engineer/Scientist → Sci-Fi/Cyberpunk
#   Music + Artist/Teacher      → Drama/Inspirational
#   Reading + Doctor/Scientist  → Mystery/Historical
#   Art + Artist/Teacher        → Creative/Fantasy
#   Nature + Doctor/Scientist   → Exploration
#   Robotics + Engineer/Scientist → Futuristic/Technology
# ─────────────────────────────────────────────────────────────────────────────

records = []

# Cricket students (dominant interest in Sri Lanka)
for i in range(30):
    aspiration = np.random.choice(['Athlete', 'Engineer', 'Teacher'], p=[0.5, 0.35, 0.15])
    records.append({'Interest': 'Cricket', 'Aspiration': aspiration,
                    'Struggle_Level': np.random.choice(['Low','Medium','High']),
                    'Story_Theme': 'Sports Adventure'})

# Gaming students
for i in range(20):
    aspiration = np.random.choice(['Engineer', 'Scientist', 'Artist'], p=[0.55, 0.30, 0.15])
    records.append({'Interest': 'Gaming', 'Aspiration': aspiration,
                    'Struggle_Level': np.random.choice(['Low','Medium','High']),
                    'Story_Theme': 'Sci-Fi/Cyberpunk'})

# Music students
for i in range(15):
    aspiration = np.random.choice(['Artist', 'Teacher', 'Doctor'], p=[0.5, 0.3, 0.2])
    records.append({'Interest': 'Music', 'Aspiration': aspiration,
                    'Struggle_Level': np.random.choice(['Low','Medium','High']),
                    'Story_Theme': 'Drama/Inspirational'})

# Reading students
for i in range(15):
    aspiration = np.random.choice(['Doctor', 'Scientist', 'Teacher'], p=[0.45, 0.35, 0.2])
    records.append({'Interest': 'Reading', 'Aspiration': aspiration,
                    'Struggle_Level': np.random.choice(['Low','Medium','High']),
                    'Story_Theme': 'Mystery/Historical'})

# Art students
for i in range(10):
    aspiration = np.random.choice(['Artist', 'Teacher', 'Engineer'], p=[0.6, 0.25, 0.15])
    records.append({'Interest': 'Art', 'Aspiration': aspiration,
                    'Struggle_Level': np.random.choice(['Low','Medium','High']),
                    'Story_Theme': 'Creative/Fantasy'})

# Nature students
for i in range(10):
    aspiration = np.random.choice(['Doctor', 'Scientist', 'Teacher'], p=[0.4, 0.4, 0.2])
    records.append({'Interest': 'Nature', 'Aspiration': aspiration,
                    'Struggle_Level': np.random.choice(['Low','Medium','High']),
                    'Story_Theme': 'Exploration'})

# Robotics students
for i in range(10):
    aspiration = np.random.choice(['Engineer', 'Scientist', 'Teacher'], p=[0.65, 0.25, 0.10])
    records.append({'Interest': 'Robotics', 'Aspiration': aspiration,
                    'Struggle_Level': np.random.choice(['Low','Medium','High']),
                    'Story_Theme': 'Futuristic/Technology'})

df = pd.DataFrame(records)
df.insert(0, 'Student_ID', range(1, len(df)+1))

# Save the CSV for reference
df.to_csv('sri_lankan_student.csv', index=False)
print(f'Dataset shape: {df.shape}')
print(df.groupby(['Interest','Story_Theme']).size().reset_index(name='count'))

In [ ]:
# ── ENCODE & TRAIN ───────────────────────────────────────────────────────────
le_interest   = LabelEncoder()
le_aspiration = LabelEncoder()
le_theme      = LabelEncoder()

X = pd.DataFrame({
    'Interest_Encoded':   le_interest.fit_transform(df['Interest']),
    'Aspiration_Encoded': le_aspiration.fit_transform(df['Aspiration']),
})
y = le_theme.fit_transform(df['Story_Theme'])

clf = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=6)
clf.fit(X, y)

# Cross-validation for honest accuracy
cv_scores = cross_val_score(clf, X, y, cv=5)
print(f'Training Accuracy : {clf.score(X, y)*100:.1f}%')
print(f'Cross-Val Accuracy: {cv_scores.mean()*100:.1f}% ± {cv_scores.std()*100:.1f}%')

# Show label mapping
print('\nTheme classes:', list(le_theme.classes_))

In [ ]:
# ── SAVE MODEL & ENCODERS ────────────────────────────────────────────────────
with open('persona_model.pkl', 'wb') as f:
    pickle.dump(clf, f)

with open('encoder.pkl', 'wb') as f:
    pickle.dump({'interest': le_interest, 'aspiration': le_aspiration, 'theme': le_theme}, f)

print('Saved persona_model.pkl and encoder.pkl')
print('Download them from the Colab file browser (left sidebar).')

In [ ]:
# ── QUICK SANITY CHECK ───────────────────────────────────────────────────────
test_cases = [
    ('Cricket', 'Engineer'),
    ('Gaming',  'Scientist'),
    ('Music',   'Artist'),
    ('Nature',  'Doctor'),
    ('Robotics','Engineer'),
]
print('Predictions:')
for interest, aspiration in test_cases:
    ie = le_interest.transform([interest])[0]
    ae = le_aspiration.transform([aspiration])[0]
    theme = le_theme.inverse_transform(clf.predict([[ie, ae]]))[0]
    print(f'  {interest:10s} + {aspiration:12s} → {theme}')